# Project — modelling notebook

**Author:** Andrea Gutiérrez Arcas** · **Dataset:**  [NBA Dataset: Box Scores and Stats (1947 - Today)](https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores) · **Last run:** *PENDIENTE*



## 0. Executive summary

> 📝 **Write this last.** Five lines for the stakeholder, no jargon:
> 1. The question the model answers, and for whom.
> 2. The model you chose, in one phrase.
> 3. How much better it is than today's rule, in the stakeholder's units.
> 4. What the stakeholder should do with it.
> 5. The main limit: when not to trust it.

*TODO*

## Setup

> 📝 Every import in one place, and the settings you may want to change. If you need a package
> that is not in the course `requirements.txt`, install it here and say why.

In [2]:
##Al ejecutar los import me ha dado error porque no tengo instalada la librería plotly.

%pip install plotly scikit-learn pyarrow

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.7 MB 15.1 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.7 MB 18.9 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 18.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

import joblib  # noqa: F401
import numpy as np  # noqa: F401
import pandas as pd
import plotly.express as px  # noqa: F401
import sklearn
from sklearn.compose import ColumnTransformer  # noqa: F401
from sklearn.dummy import DummyClassifier, DummyRegressor  # noqa: F401
from sklearn.ensemble import (  # noqa: F401
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer  # noqa: F401
from sklearn.inspection import permutation_importance  # noqa: F401
from sklearn.linear_model import LinearRegression, LogisticRegression  # noqa: F401
from sklearn.metrics import (  # noqa: F401
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (  # noqa: F401
    GridSearchCV,
    RandomizedSearchCV,
    TimeSeriesSplit,
    cross_val_score,
)
from sklearn.pipeline import Pipeline  # noqa: F401
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # noqa: F401

# Use RANDOM_STATE wherever a random_state exists, so two runs give the same numbers.
RANDOM_STATE = 42
# Relative to this notebook (project/), or a URL string.
DATA_PATH = Path("data") / "nba_load.parquet" #ruta a mi notebook de limpieza
MODEL_PATH = Path("model.joblib")

print("pandas", pd.__version__, "· scikit-learn", sklearn.__version__)

pandas 3.0.1 · scikit-learn 1.9.1


## 1. From the brief to an ML task

> 📝 Translate your corrected brief into an ML task before writing any model code (brief §2).
> Fill in the table. If the target, a hypothesis or a KPI changed since the brief, say what
> changed and why below it.

| Question | Your answer |
|---|---|
| Decision the prediction feeds | *TODO* |
| Task type (regression / classification) | *TODO* |
| Target: exact column or formula | *TODO* |
| Grain: what is one prediction? | *TODO* |
| When is the prediction made, and how far ahead? | *TODO* |
| Features available at that moment | *TODO* |
| Today's rule (the baseline to beat) | *TODO* |
| Model metric, and why | *TODO* |
| Business translation (which KPI moves) | *TODO* |

**Changes since the brief:** *TODO — or "none".*

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 2. Data and quality

> 📝 Load the data, recap the dataset card in three or four lines (publisher, licence, grain,
> period), and apply the cleaning from lesson 03. Report it **with numbers**: rows before and
> after, nulls filled or dropped, duplicates removed, outliers treated and how.

In [ ]:
# TODO load the raw data
# raw = pd.read_csv(DATA_PATH, parse_dates=["TODO_date_column"])
# raw.shape, raw.dtypes

In [ ]:
# TODO clean it, and keep the counts you will report
# df = raw.copy()
# n_before = len(df)
# df = df.drop_duplicates()
# ...
# print(f"{n_before - len(df)} rows removed")

| Step | Rows affected | Decision and reason |
|---|---|---|
| *TODO* | | |

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 3. What the exploration showed

> 📝 Show the target's distribution and two to four charts that matter for the model (plotly,
> titled, axes labelled with units). Then revisit every hypothesis of your brief with the
> evidence you now have.

In [ ]:
# TODO the target's distribution
# px.histogram(df, x="TODO_target", title="TODO", labels={"TODO_target": "TODO (units)"})

In [ ]:
# TODO two to four charts that matter for the model

| Hypothesis (from the brief) | Status: supported / refuted / not testable yet | Evidence (chart, table or number) |
|---|---|---|
| H1 — *TODO* | | |
| H2 — *TODO* | | |

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 4. Features

> 📝 Build the features (lesson 05: lags, rolling means, date parts, group aggregates,
> encodings) and fill in the feature table. Then the **leakage check**: for every feature, would
> its value be known at the moment the prediction is made?

In [ ]:
# TODO engineer the features
# df["month"] = df["TODO_date_column"].dt.month
# df["TODO_lag1"] = df.groupby("TODO_key")["TODO_target"].shift(1)

In [ ]:
# TODO the final lists the pipeline will use
TARGET = "TODO_target"
NUMERIC = []  # e.g. ["TODO_lag1", "month"]
CATEGORICAL = []  # e.g. ["TODO_district"]

| Feature | How it is built | Why it should help | Known at prediction time? |
|---|---|---|---|
| *TODO* | | | yes / no → dropped |

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 5. Split and baseline

> 📝 If your data has dates, split in time: train on the past, validate on the next period, and
> keep the most recent period as the **test set, untouched until section 8**. Score today's
> rule and a naive baseline (mean, median or majority class) on the validation data: every
> model from now on is judged against them.

In [ ]:
# TODO the split — dates you choose and why
# train = df[df["TODO_date_column"] < "TODO"]
# valid = df[(df["TODO_date_column"] >= "TODO") & (df["TODO_date_column"] < "TODO")]
# test = df[df["TODO_date_column"] >= "TODO"]
# X_train, y_train = train[NUMERIC + CATEGORICAL], train[TARGET]
# X_valid, y_valid = valid[NUMERIC + CATEGORICAL], valid[TARGET]

In [ ]:
# TODO today's rule and a naive baseline, scored on the validation data
# rule_pred = valid["TODO_column_the_rule_uses"]
# naive = DummyRegressor(strategy="median").fit(X_train, y_train)  # or DummyClassifier
# mean_absolute_error(y_valid, rule_pred), mean_absolute_error(y_valid, naive.predict(X_valid))

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 6. Candidate models

> 📝 At least **two** models besides the baselines: one simple (linear or logistic regression,
> a shallow tree) and one ensemble (random forest or gradient boosting). Build each as a
> pipeline, so the preprocessing is fitted on the training data only. Finish with one
> comparison table on the validation data, baselines included.

In [ ]:
# TODO the preprocessing shared by every model
# preprocess = ColumnTransformer([
#     ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
#                       ("scale", StandardScaler())]), NUMERIC),
#     ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
# ])

In [ ]:
# TODO the candidates
# candidates = {
#     "linear": Pipeline([("prep", preprocess), ("model", LinearRegression())]),
#     "random forest": Pipeline([("prep", preprocess),
#                                ("model", RandomForestRegressor(random_state=RANDOM_STATE))]),
# }

In [ ]:
# TODO fit each one and build the comparison table
# rows = []
# for name, pipe in candidates.items():
#     pipe.fit(X_train, y_train)
#     pred = pipe.predict(X_valid)
#     rows.append({"model": name, "MAE": mean_absolute_error(y_valid, pred)})
# comparison = pd.DataFrame(rows)
# comparison

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 7. Metric and cost of errors

> 📝 Why is your main metric the right one for this decision? What does one error cost the
> stakeholder?
> - **Classification:** the confusion matrix read in business terms (what a false positive and
>   a false negative cost), and the threshold you chose.
> - **Regression:** MAE against RMSE, and where the large errors fall (which segments, which
>   periods).

In [ ]:
# TODO the error analysis for the best candidate

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 8. Validation and tuning

> 📝 Cross-validate the chosen model (`TimeSeriesSplit` when your data has dates), tune it with
> a grid or random search, and only then evaluate it **once** on the test set. Report the
> score per fold, not only the mean: a model that is good on average but erratic is a risk.

In [ ]:
# TODO cross-validation and the search
# cv = TimeSeriesSplit(n_splits=5)
# search = GridSearchCV(candidates["TODO"], param_grid={"model__max_depth": [4, 8, None]},
#                       cv=cv, scoring="neg_mean_absolute_error")
# search.fit(pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid]))
# search.best_params_, -search.cv_results_["mean_test_score"]

In [ ]:
# TODO the one final evaluation on the test set
# final_model = search.best_estimator_
# X_test, y_test = test[NUMERIC + CATEGORICAL], test[TARGET]
# mean_absolute_error(y_test, final_model.predict(X_test))

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 9. Interpretation

> 📝 Which features drive the predictions? Use permutation importance (or SHAP if you want to
> go further). Then explain two or three individual predictions: why did the model say this for
> this row? Do the drivers match your hypotheses?

In [ ]:
# TODO feature importance
# result = permutation_importance(final_model, X_test, y_test, n_repeats=10,
#                                 random_state=RANDOM_STATE)
# importance = pd.Series(result.importances_mean, index=X_test.columns).sort_values()
# px.bar(importance, orientation="h", title="TODO")

In [ ]:
# TODO two or three individual predictions, explained

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 10. Fairness, ethics and risks

> 📝 Compare the errors across at least one meaningful group (district, segment, period): is the
> model worse for some of them? Then answer in prose:
> - Does the data contain personal or sensitive information? How is it protected (lesson 03:
>   GDPR, anonymisation, access)?
> - What happens if the model is wrong, and who bears the cost?
> - One mitigation for each risk you name.

In [ ]:
# TODO the error per group
# errors = test.assign(abs_error=(y_test - final_model.predict(X_test)).abs())
# errors.groupby("TODO_group")["abs_error"].mean()

**What this means for the stakeholder.** *TODO — two or three sentences in plain language, no
jargon. This is the text you will lift into your AB.*

## 11. Recommendation

> 📝 Speak to the stakeholder:
> - What should they do with the model, starting when?
> - The expected improvement over today's rule, translated into a KPI of your brief or into
>   money.
> - When it should **not** be trusted (periods, segments, data changes).
> - What block 4 adds: how it will be deployed and monitored.

*TODO*

## 12. Saving the model

> 📝 Save the final pipeline — from raw columns to a prediction — and prove it works by reloading
> it and predicting one row. This is the file lesson 10 deploys. Commit it if it is under 50 MB.

In [ ]:
# TODO save, reload, predict one row
# joblib.dump(final_model, MODEL_PATH)
# reloaded = joblib.load(MODEL_PATH)
# reloaded.predict(X_test.head(1))

## 13. References

> 📝 Every source you used — the dataset, articles, documentation — in **APA** format. A missing
> or incorrect reference can cost the Pass.

- *TODO* Publisher. (Year). *Dataset title* [Data set]. URL